This code is partly base on Marina's work.
Used for analyse the WACCM data for several significant years.

In [ ]:
from netCDF4 import Dataset, MFDataset
import numpy as np
from ozone_extremes_leap import *
import matplotlib.pyplot as plt
import xarray as xr 
from mpl_toolkits.basemap import Basemap
from matplotlib.pyplot import cm
import seaborn as sns
from matplotlib import rc
import warnings
from Weiji_module import *
import os

导入WACCM模拟timeslice 2000 dataset
1.		选取60-90°N,并计算cos(lat)加权的经向-纬向平均
•	使用之前的calc_parc_o3计算柱浓度
•	经度平均
•	选取60-90N纬度范围
•	用cos(lat)加权平均得到区域平均
2.	海平面气压(PSL)处理:
•	计算日气候平均(按一年中的日期分组求平均)

In [ ]:
#WACCM INT_O3 PSL
nc_fid_SURF=xr.open_dataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.PSL.merged.nc')
PSL_clim=nc_fid_SURF['PSL']
PSL_clim=PSL_clim.groupby('time.dayofyear').mean() 

#WACCM INT_O3
nc_fid=xr.open_dataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc')  

O3=nc_fid['O3']

# 计算两种臭氧柱
O3_30_70 = calc_parc_o3(O3, 30, 70)
O3_30_100 = calc_parc_o3(O3, 30, 100)

def spatial_average(O3_data, lat_start=60, lat_end=90):
    """
    计算臭氧数据的空间平均
    
    参数:
    O3_data: xarray.DataArray - 输入的臭氧数据
    lat_start: float - 起始纬度，默认60
    lat_end: float - 结束纬度，默认90
    
    返回:
    xarray.DataArray: 空间平均后的数据
    """
    O3_avg = O3_data.mean(dim='lon')
    var = O3_avg.sel(lat=slice(lat_start, lat_end))
    weights = np.cos(np.deg2rad(var.lat))
    var = var.weighted(weights)     
    return var.mean(dim='lat')

# 计算30-70hPa的热带和极圈平均
var_30_70_tropical = spatial_average(O3_30_70, lat_start=0, lat_end=30)  # 热带区域
var_30_70_polar = spatial_average(O3_30_70, lat_start=60, lat_end=90)    # 极圈区域

# 计算30-100hPa的热带和极圈平均
var_30_100_tropical = spatial_average(O3_30_100, lat_start=0, lat_end=30)  # 热带区域
var_30_100_polar = spatial_average(O3_30_100, lat_start=60, lat_end=90)    # 极圈区域

找极值然后绘制了北极地区臭氧随时间的演变:
1.	通过调用find_ozone_extremes和analyse_ozone_extremes找出臭氧最低的年
(1)	find_ozone_extremes30100
	计算60-90N纬度带30-100hPa的臭氧柱浓度 
	选取3-4月数据 
	找出每年春季臭氧的最大值和最小值及其发生日期(直接groupby("time.year").max/min()找) 
	返回极值年份的索引、日期、发生时间等信息

(2)	analyse_ozone_extremes
	 计算气压场(PSL)的日距平（某一天的值减去该日期在所有年份的平均值(如3月1日的值减去所有年份3月1日的平均值)，直接用var.groupby("time.dayofyear") 减去 var.groupby("time.dayofyear").mean("time")）
	 提取臭氧极值日期前后60天的气压场数据 
	 对这60天数据求平均 
	 对结果进行插值(特别是极地地区) 插值调用了interpolate_pole和interpolate_lon
a 	对于pole ：在纬度方向上添加90°N和90°S 经度方向上添加360°(等于0°)然后通过计算离极点最近纬度圈的平均值作为新格点的填充值(另：360°经度取0°的值）
b 	对于lon只在经度方向添加360°点（直接取0°的值）
	 返回最低臭氧年的平均气压场响应及绘图所需坐标（将经纬度网格转换为地图投影坐标，方便绘图）


更换到新的子程序计算新的30-100和以前的30-70hpa的臭氧住浓度

In [ ]:
def plot_o3_case(var, case='30-70', region='polar', extreme_years=10 ,years=39, model='WACCM', lev='plev'):
    """
    绘制臭氧演变图
    参数:
    region: str - 'tropical' 或 'polar'，用于标识区域
    """
    fig, ax = plt.subplots(1,1,figsize=(13,10), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')
    
    O3 = 'O3'
    # 从case中提取压力层范围
    p_range = case.split('-')
    p_top = int(p_range[0])
    p_bottom = int(p_range[1])
   
    # 使用新的函数，传入压力层参数
    O3_highest, O3_lowest, O3_lowest_date, O3_highest_date, O3_lowest_index, O3_highest_index, O3_intersect, O3_years = find_ozone_extremes_baseonpressure(nc_fid, O3, lev, years, extreme_years, model, p_top=p_top, p_bottom=p_bottom)
    
    
    # 地表气压分析保持不变
    var_min_slp_x, xpt, ypt, var_anomalies_slp_x, significance_slp_x = analyse_ozone_extremes(nc_fid_SURF, 'PSL', extreme_years, True, O3_highest_date, O3_lowest_date, O3_lowest_index, O3_years, model)

    # 计算中性年份
    O3_neutral_years = np.zeros((years-extreme_years))
    x = 0
    O3_lowest = O3_years[O3_lowest]
    
    for y in range(years):
        if np.isin(O3_years[y], O3_lowest) == False:
            O3_neutral_years[x] = O3_years[y]
            x = x + 1

    # 统计计算部分
    var_clim = var.groupby("time.dayofyear").mean("time")
    
    var_lowest = var.sel(time=var.time.dt.year.isin([O3_lowest]))
    var_lowest_mean = np.array(var_lowest.groupby("time.dayofyear").mean("time"))
    var_lowest_std = np.array(var_lowest.groupby("time.dayofyear").std("time"))
    
    var_lowest_before = var.sel(time=var.time.dt.year.isin([O3_lowest-1]))
    var_lowest_mean_before = np.array(var_lowest_before.groupby("time.dayofyear").mean("time"))
    var_lowest_std_before = np.array(var_lowest_before.groupby("time.dayofyear").std("time"))
    
    var_neutral = var.sel(time=var.time.dt.year.isin([O3_neutral_years]))
    var_neutral_mean = np.array(var_neutral.groupby("time.dayofyear").mean("time"))
    var_neutral_std = np.array(var_neutral.groupby("time.dayofyear").std("time"))
    
    var_neutral_before = var.sel(time=var.time.dt.year.isin([O3_neutral_years-1]))
    var_neutral_mean_before = np.array(var_neutral_before.groupby("time.dayofyear").mean("time"))
    var_neutral_std_before = np.array(var_neutral_before.groupby("time.dayofyear").std("time"))

    years_list = O3_lowest.tolist()  # 转换为列表
    var_lowest = xr.concat([var.sel(time=var.time.dt.year == year) for year in years_list], dim='time')
    var_lowest_before = xr.concat([var.sel(time=var.time.dt.year == (year - 1)) for year in years_list], dim='time')

    # reshape数据
    var_lowest = np.reshape(np.array(var_lowest), (extreme_years, 365))
    var_lowest_before = np.reshape(np.array(var_lowest_before), (extreme_years, 365))

    # 绘图部分
    color = cm.twilight(np.linspace(0, 1, 15))
    
    # 创建一个空的图例标签列表
    legend_labels = []
    
    for j in range(extreme_years):
        # 获取当前年份
        current_year = int(O3_lowest[j])
        # 创建该年的图例标签
        label = f'Low O3 year {current_year}'
        
        # 绘制曲线并添加标签
        ax.plot(range(91,365), var_lowest[j,0:365-91], 
                color=color[j], alpha=0.8, linewidth=2, 
                label=label)
        ax.plot(range(0,91), var_lowest_before[j,365-92:365-1], 
                color=color[j], alpha=0.8, linewidth=2)
        
        legend_labels.append(label)
    
    # 绘制中性年份的曲线
    plot = ax.errorbar(range(91,365), var_neutral_mean[0:365-91], var_neutral_std[0:365-91], 
                      color='grey', alpha=0.5, elinewidth=1.5, linewidth=3, 
                      label='All other years')
    
    ax.errorbar(range(0,91), var_neutral_mean_before[365-92:365-1], var_neutral_std[365-92:365-1], 
                color='grey', alpha=0.5, elinewidth=1.5, linewidth=3)
    
    # 添加图例
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12)
    # 设置图形属性
    ax.set_xlim(0,366)
    ax.set_xticks([0,31,61,91,122,150,181,211,242,273,304,334])
    ax.set_xticklabels(['Oct', 'Nov', 'Dec', 'Jan','Feb','Mar','Apr','May','June','July','Aug','Sep'], fontsize=15)
    # 修改标题和标签以反映区域
    region_name = 'Tropical' if region == 'tropical' else 'Polar'
    ax.set_ylabel(f'Partial ozone column ({region_name}, {case} hPa) (DU)', fontsize=18)
    ax.set_yticklabels(ax.get_yticks(), size = 18)
    
    # 保存图形时加入区域信息
    plt.savefig(f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_10extr_{case.replace("-","to")}_{region}.pdf')
    plt.savefig(f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_10extr_{case.replace("-","to")}_{region}.png')
    
    return fig, ax

fig_30_70_polar10 = plot_o3_case(var_30_70_polar, case='30-70', region='polar', extreme_years=10)
#fig_30_70_polar5 = plot_o3_case(var_30_70_polar, case='30-70', region='polar', extreme_years=5)
fig_30_100_polar10 = plot_o3_case(var_30_100_polar, case='30-100', region='polar', extreme_years=10)
#fig_30_100_polar5 = plot_o3_case(var_30_100_polar, case='30-100', region='polar', extreme_years=5)

#fig_30_70_tropica10 = plot_o3_case(var_30_70_tropical, case='30-70', region='tropical', extreme_years=10)
#fig_30_70_tropica5 = plot_o3_case(var_30_70_tropical, case='30-70', region='tropical', extreme_years=5)
#fig_30_100_tropical0 = plot_o3_case(var_30_100_tropical, case='30-100', region='tropical', extreme_years=10)
#fig_30_100_tropica5 = plot_o3_case(var_30_100_tropical, case='30-100', region='tropical', extreme_years=5)

绘制地面响应

In [ ]:
def plot_surface_response(nc_fid_SURF, nc_fid, case='30-70', extreme_years=10, years=39, model='WACCM', lev='plev'):
   """
   绘制地表响应图
   
   参数:
   nc_fid_SURF: 包含地表数据的文件
   nc_fid: 包含臭氧数据的文件
   case: 压力层范围标识，如'30-70'或'30-100'  
   extreme_years: 极值年份数量
   years: 总年份数
   model: 模式名称
   lev: 压力层变量名
   """
   # 从case提取压力层范围
   p_range = case.split('-')
   p_top = int(p_range[0])
   p_bottom = int(p_range[1])
   
   # 获取臭氧极值年份信息
   O3_highest, O3_lowest, O3_lowest_date, O3_highest_date, O3_lowest_index, O3_highest_index, O3_intersect, O3_years = find_ozone_extremes_baseonpressure(nc_fid, 'O3', lev, years, extreme_years, model, p_top=p_top, p_bottom=p_bottom)
   
   # 分析地表响应
   var_min_slp, xpt, ypt, var_anomalies_slp, significance_slp = analyse_ozone_extremes(nc_fid_SURF, 'PSL', extreme_years, True, O3_highest_date, O3_lowest_date, O3_lowest_index, O3_years, model)
   
   # 绘图
   fig = plt.figure(figsize=(10,8))
   
   m = Basemap(projection='ortho', lat_0=90, lon_0=0, resolution='l')
   m.drawcoastlines(linewidth=0.3)
   m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
   m.drawparallels(np.arange(-90,90,30), linewidth=0.3)

   levels = np.linspace(-4,4, 16)
   plot_slp = m.contourf(xpt, ypt, var_min_slp/100, 
                        cmap=plt.cm.get_cmap('RdBu_r'), 
                        interpolation='gaussian', 
                        levels=levels, 
                        extend='both')

   plt.title(f'Mean surface response ({case} hPa, n={extreme_years})')
   
   plt.colorbar(plot_slp)
   
   suffix = f"_{O3_years[O3_lowest[0]]}" if extreme_years == 1 else f"_n{extreme_years}"
   plt.savefig(f'/home/weiji/restart_exam/plots/surface_response_{case.replace("-","to")}{suffix}.pdf')
   plt.savefig(f'/home/weiji/restart_exam/plots/surface_response_{case.replace("-","to")}{suffix}.png')
   
   return fig

# 使用示例：
# 分别绘制30-70和30-100的图
fig_30_70 = plot_surface_response(nc_fid_SURF, nc_fid, case='30-70')
fig_30_100 = plot_surface_response(nc_fid_SURF, nc_fid, case='30-100')

In [ ]:
# 单个年份的30-70情况
fig_single_30_70 = plot_surface_response(nc_fid_SURF, nc_fid, case='30-70', extreme_years=1)

# 单个年份的30-100情况
fig_single_30_100 = plot_surface_response(nc_fid_SURF, nc_fid, case='30-100', extreme_years=1)

In [ ]:
# 对30-70hPa的数据处理
# 热带区域
var_30_70_tropical_select = var_30_70_tropical.sel(time=var_30_70_tropical.time.dt.month.isin([1,2,3,4,5,6]))
var_0008_30_70_tropical = var_30_70_tropical_select.sel(time=var_30_70_tropical_select.time.dt.year.isin([8]))
var_clim_30_70_tropical = var_30_70_tropical_select.groupby('time.dayofyear').mean()

# 极圈区域
var_30_70_polar_select = var_30_70_polar.sel(time=var_30_70_polar.time.dt.month.isin([1,2,3,4,5,6]))
var_0008_30_70_polar = var_30_70_polar_select.sel(time=var_30_70_polar_select.time.dt.year.isin([8]))
var_clim_30_70_polar = var_30_70_polar_select.groupby('time.dayofyear').mean()

# 对30-100hPa的数据处理
# 热带区域
var_30_100_tropical_select = var_30_100_tropical.sel(time=var_30_100_tropical.time.dt.month.isin([1,2,3,4,5,6]))
var_0008_30_100_tropical = var_30_100_tropical_select.sel(time=var_30_100_tropical_select.time.dt.year.isin([8]))
var_clim_30_100_tropical = var_30_100_tropical_select.groupby('time.dayofyear').mean()

# 极圈区域
var_30_100_polar_select = var_30_100_polar.sel(time=var_30_100_polar.time.dt.month.isin([1,2,3,4,5,6]))
var_0008_30_100_polar = var_30_100_polar_select.sel(time=var_30_100_polar_select.time.dt.year.isin([8]))
var_clim_30_100_polar = var_30_100_polar_select.groupby('time.dayofyear').mean()

30-70和30-100分别绘图

In [ ]:
def process_o3_data(file_pattern, prefix):
    """
    处理臭氧数据的通用函数
    
    Parameters:
    -----------
    file_pattern : str
        文件路径模式
    prefix : str
        变量前缀（如 'J', 'F', 'M' 等）
    """
    # 读取数据
    nc_fid = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    
    # 获取并处理臭氧数据
    o3 = nc_fid['O3']
    o3 = o3.mean(dim='lon')
    
    # 设置全局变量名
    globals()[f'o3_{prefix}'] = o3
    
    # 处理30-70hPa的数据
    o3_30_70 = calc_parc_o3(o3, 30, 70)
    globals()[f'o3_{prefix}_30_70'] = o3_30_70
    
    var_30_70 = o3_30_70.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(var_30_70.lat))
    var_30_70 = var_30_70.weighted(weights)
    var_30_70 = var_30_70.mean(dim='lat')
    globals()[f'var_{prefix}_30_70'] = var_30_70
    
    # 处理30-100hPa的数据
    o3_30_100 = calc_parc_o3(o3, 30, 100)
    globals()[f'o3_{prefix}_30_100'] = o3_30_100
    
    var_30_100 = o3_30_100.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(var_30_100.lat))
    var_30_100 = var_30_100.weighted(weights)
    var_30_100 = var_30_100.mean(dim='lat')
    globals()[f'var_{prefix}_30_100'] = var_30_100

# 基础路径
base_path = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008'

# 定义所有需要处理的数据集
# FNC是nocoupl
datasets = {
    'F': 'Feb/BWCN.e122.f19_g16.002.0008-02.*.nc',
    'F2': 'Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc',
    'F3': 'Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc',
    'FNC': 'Feb_NOCOUPL/BWCN.e122.f19_g16.002.0008-02.*.nc',
    'M': 'Mar/BWCN.e122.f19_g16.002.0008-03.*.nc',
    'M2': 'Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc',
    'M3': 'Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc',
    'A2': 'Apr/BWCN.e122.f19_g16.002.0008-04.*.nc',
    'J': 'Jan/BWCN.e122.f19_g16.002.0008-01.*.nc'
}

# 处理所有数据集
for prefix, file_path in datasets.items():
    full_path = f'{base_path}/{file_path}'
    process_o3_data(full_path, prefix)



In [ ]:
def plot_o3_evolution(var_0008, var_clim, var_J, var_F, var_M, pressure_range='30-70'):
    """
    绘制臭氧演变图。
    - 基准曲线(var_0008)和气候态(var_clim)均以黑色绘制（实线和虚线）。
    - Jan、Feb、Mar数据仅绘制均值曲线，并用每个时间点的最小值到最大值包络区作为阴影显示。
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("ticks")
    
    fig, ax = plt.subplots(1, 1, figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    # 绘制基准和气候态（均为黑色）
    x_base = range(len(var_0008))
    ax.plot(x_base, var_0008, color='black', linewidth=3, label='Base')
    x_clim = range(len(var_clim))
    ax.plot(x_clim, var_clim, color='black', linestyle=':', linewidth=3, label='Clim')
    
    # Jan数据：起始位置为0
    x_J = range(len(var_J.time))
    mean_J = var_J.mean(dim='member')
    env_min_J = var_J.min(dim='member')
    env_max_J = var_J.max(dim='member')
    ax.plot(x_J, mean_J, color='forestgreen', linewidth=3, label='Jan small pertlim')
    ax.fill_between(x_J, env_min_J, env_max_J, color='forestgreen', alpha=0.3)
    
    # Feb数据：起始位置为31
    x_F = range(31, 31 + len(var_F.time))
    mean_F = var_F.mean(dim='member')
    env_min_F = var_F.min(dim='member')
    env_max_F = var_F.max(dim='member')
    ax.plot(x_F, mean_F, color='royalblue', linewidth=3, label='Feb small pertlim')
    ax.fill_between(x_F, env_min_F, env_max_F, color='royalblue', alpha=0.3)
    
    # Mar数据：起始位置为59
    x_M = range(59, 59 + len(var_M.time))
    mean_M = var_M.mean(dim='member')
    env_min_M = var_M.min(dim='member')
    env_max_M = var_M.max(dim='member')
    ax.plot(x_M, mean_M, color='hotpink', linewidth=3, label='Mar small pertlim')
    ax.fill_between(x_M, env_min_M, env_max_M, color='hotpink', alpha=0.3)
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=22)
    ax.set_ylabel(f'Partial ozone column, {pressure_range} hPa (DU)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    plt.legend(fontsize=16)
    
    plt.savefig(f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_0008_{pressure_range.replace("-","to")}.pdf')
    plt.savefig(f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_0008_{pressure_range.replace("-","to")}.png')
    
    return fig, ax

# 绘制30-70hPa的图
fig_30_70 = plot_o3_evolution(var_0008_30_70_polar, var_clim_30_70_polar, 
                            var_J_30_70, var_F_30_70, var_M_30_70, 
                            pressure_range='30-70')

# 绘制30-100hPa的图
#fig_30_100 = plot_o3_evolution(var_0008_30_100_polar, var_clim_30_100_polar, 
#                              var_J_30_100, var_F_30_100, var_M_30_100, 
#                              pressure_range='30-100')

In [ ]:
# 读取地表数据 - Feb
nc_fid_F = xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_SURF/BWCN.e122.f19_g16.002.0008-02.*.nc', concat_dim='member', combine='nested')
p_F = nc_fid_F['PSL']

# 读取地表数据 - Feb NOCOUPL
nc_fid_FNC = xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL_SURF/BWCN.e122.f19_g16.002.0008-02.*.nc', concat_dim='member', combine='nested')
p_FNC = nc_fid_FNC['PSL']

# 分别处理30-70hPa和30-100hPa的数据
for case in ['30-70', '30-100']:
    # 选择相应的var数据
    if case == '30-70':
        var_F = var_F_30_70
        var_FNC = var_FNC_30_70
    else:
        var_F = var_F_30_100
        var_FNC = var_FNC_30_100
        
    # 处理Feb数据
    var_F_runmean = var_F.rolling(time=5, center=True).mean()
    var_F_runmean = var_F_runmean.sel(time=var_F_runmean.time.dt.month.isin([3,4]))
    var_F_runmean = var_F_runmean.groupby('member').argmin(dim='time')
    print(f"{case} Feb min dates:", np.array(var_F_runmean))
    
    # 处理Feb NOCOUPL数据
    var_FNC_runmean = var_FNC.rolling(time=5, center=True).mean()
    var_FNC_runmean = var_FNC_runmean.sel(time=var_FNC_runmean.time.dt.month.isin([3,4]))
    var_FNC_runmean = var_FNC_runmean.groupby('member').argmin(dim='time')
    print(f"{case} Feb NOCOUPL min dates:", np.array(var_FNC_runmean))
    
    # 计算Feb异常
    p_F_anom = p_F.groupby("time.dayofyear") - PSL_clim
    p_F_time = p_F_anom.sel(time=p_F_anom.time.dt.month.isin([3,4,5]))
    p_F_time = p_F_time.groupby('member')
    
    PSL_results = []
    for i, (member, group) in enumerate(p_F_time):
        PSL_results.append(np.array(np.mean(group[int(var_F_runmean[i]):int(var_F_runmean[i])+30,:,:], axis=0)))
    
    if case == '30-70':
        PSL_30_70 = np.nanmean(PSL_results, axis=0)
    else:
        PSL_30_100 = np.nanmean(PSL_results, axis=0)
    
    # 计算Feb NOCOUPL异常
    p_FNC_anom = p_FNC.groupby("time.dayofyear") - PSL_clim
    p_FNC_time = p_FNC_anom.sel(time=p_FNC_anom.time.dt.month.isin([3,4,5]))
    p_FNC_time = p_FNC_time.groupby('member')
    
    PSL_NC_results = []
    for i, (member, group) in enumerate(p_FNC_time):
        PSL_NC_results.append(np.array(np.mean(group[int(var_FNC_runmean[i]):int(var_FNC_runmean[i])+30,:,:], axis=0)))
    
    if case == '30-70':
        PSL_NC_30_70 = np.nanmean(PSL_NC_results, axis=0)
    else:
        PSL_NC_30_100 = np.nanmean(PSL_NC_results, axis=0)

In [ ]:
# 对于两种压力层范围分别处理数据
def process_surface_data(nc_fid_surf, var_data, PSL_clim, case='30-70'):
    """
    处理地表数据的函数
    
    Parameters:
    -----------
    nc_fid_surf: xarray Dataset
        包含地表数据的文件
    var_data: xarray DataArray
        臭氧数据
    PSL_clim: xarray DataArray
        气候态数据
    case: str
        压力层范围 ('30-70' 或 '30-100')
    """
    p_data = nc_fid_surf['PSL']
    
    # 计算5天滑动平均
    var_runmean = var_data.rolling(time=5, center=True).mean()
    var_runmean = var_runmean.sel(time=var_runmean.time.dt.month.isin([3,4]))
    var_runmean = var_runmean.groupby('member').argmin(dim='time')
    
    # 移除日气候态得到异常
    p_anomaly = p_data.groupby("time.dayofyear") - PSL_clim
    p_time = p_anomaly.sel(time=p_anomaly.time.dt.month.isin([3,4,5]))
    p_time = p_time.groupby('member')
    
    # 计算臭氧最小值后30天的平均
    PSL_result = []
    for i, (member, group) in enumerate(p_time):
        PSL_result.append(np.array(
            np.mean(group[int(var_runmean[i]):int(var_runmean[i])+30,:,:], 
            axis=0)))
    
    return np.nanmean(PSL_result, axis=0)

# 处理两个压力层范围的数据
PSL_30_70 = process_surface_data(nc_fid_F, var_F_30_70, PSL_clim, case='30-70')
PSL_30_100 = process_surface_data(nc_fid_F, var_F_30_100, PSL_clim, case='30-100')

# 处理NOCOUPL数据
PSL_NC_30_70 = process_surface_data(nc_fid_FNC, var_FNC_30_70, PSL_clim, case='30-70')
PSL_NC_30_100 = process_surface_data(nc_fid_FNC, var_FNC_30_100, PSL_clim, case='30-100')

# 绘图函数
def plot_surface_responses(var_min_slp_x, var_min_slp_008, PSL_data, PSL_NC_data, case='30-70'):
    """
    绘制地表响应图
    """
    fig, ax = plt.subplots(1,4,figsize=(12,4), constrained_layout=True, 
                          edgecolor='k', sharex='col', sharey='row')
    # 添加主标题
    fig.suptitle(f'Surface Response (Minimum dates for {case} hPa)', fontsize=20, y=1.1)
    levels = np.linspace(-6,6, 25)
    
    # 绘制四个子图
    for i, (data, title) in enumerate([
        (var_min_slp_x, f'mean surface response \n after ozone minima \n (n=10)'),
        (var_min_slp_008, f'surface response \n in year 008 \n (n=1)'),
        (PSL_data, f'surface response, Feb, \n small pertlim \n (n=30)'),
        (PSL_NC_data, f'surface response, Feb, \n NOCOUPL \n (n=30)')
    ]):
        m = Basemap(projection='ortho', lat_0=90, lon_0=0, resolution='l', ax=ax[i])
        m.drawcoastlines(linewidth=0.3)
        m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
        m.drawparallels(np.arange(-90,90,30), linewidth=0.3)
        
        if i < 2:
            plot_slp = m.contourf(xpt, ypt, data/100, 
                                cmap=plt.cm.get_cmap('RdBu_r'),
                                interpolation='gaussian',
                                levels=levels, extend='both')
        else:
            lons_m, lats_m = np.meshgrid(p_F.lon, p_F.lat)
            xpt2, ypt2 = m(lons_m, lats_m)
            plot_slp = m.contourf(xpt2, ypt2, data/100,
                                cmap=plt.cm.get_cmap('RdBu_r'),
                                interpolation='gaussian',
                                levels=levels, extend='both')
        
        ax[i].set_title(title, fontsize=18)
    
    # 添加colorbar
    cbar = fig.colorbar(plot_slp, ax=ax[0:4], location='bottom', shrink=0.5)
    cbar.set_label(r"hPa", fontsize=18)
    
    # 保存图片
    plt.savefig(f'/home/weiji/restart_exam/plots/slp_response_ozone_min_Feb_{case.replace("-","to")}.pdf', 
                bbox_inches="tight")
    
    return fig, ax
# 定义全局变量
lev = 'plev'  # 压力层变量名
years = 39    # 总年数
extreme_years = 10  # 极值年份数
model = 'WACCM'  # 模式名称

# 获取两个压力层范围的极值年份
O3_highest_30_70, O3_lowest_30_70, O3_lowest_date_30_70, O3_highest_date_30_70, O3_lowest_index_30_70, O3_highest_index_30_70, O3_intersect_30_70, O3_years = find_ozone_extremes_baseonpressure(nc_fid, 'O3', lev, years, extreme_years, model, p_top=30, p_bottom=70)

O3_highest_30_100, O3_lowest_30_100, O3_lowest_date_30_100, O3_highest_date_30_100, O3_lowest_index_30_100, O3_highest_index_30_100, O3_intersect_30_100, O3_years = find_ozone_extremes_baseonpressure(nc_fid, 'O3', lev, years, extreme_years, model, p_top=30, p_bottom=100)

# 计算两个压力层范围的地表响应
var_min_slp_x_30_70, xpt, ypt, var_anomalies_slp_x_30_70, significance_slp_x_30_70 = analyse_ozone_extremes(
    nc_fid_SURF, 'PSL', extreme_years, True, 
    O3_highest_date_30_70, O3_lowest_date_30_70, 
    O3_lowest_index_30_70, O3_years, model
)

var_min_slp_x_30_100, _, _, var_anomalies_slp_x_30_100, significance_slp_x_30_100 = analyse_ozone_extremes(
    nc_fid_SURF, 'PSL', extreme_years, True, 
    O3_highest_date_30_100, O3_lowest_date_30_100, 
    O3_lowest_index_30_100, O3_years, model
)

# 计算单个年份(year 008)的响应
var_min_slp_008_30_70, _, _, var_anomalies_slp_008_30_70, significance_slp_008_30_70 = analyse_ozone_extremes(
    nc_fid_SURF, 'PSL', 1, True, 
    O3_highest_date_30_70, O3_lowest_date_30_70, 
    O3_lowest_index_30_70, O3_years, model
)

var_min_slp_008_30_100, _, _, var_anomalies_slp_008_30_100, significance_slp_008_30_100 = analyse_ozone_extremes(
    nc_fid_SURF, 'PSL', 1, True, 
    O3_highest_date_30_100, O3_lowest_date_30_100, 
    O3_lowest_index_30_100, O3_years, model
)


# 然后再使用之前的绘图函数
fig_30_70 = plot_surface_responses(var_min_slp_x_30_70, var_min_slp_008_30_70, 
                                 PSL_30_70, PSL_NC_30_70, case='30-70')
fig_30_100 = plot_surface_responses(var_min_slp_x_30_100, var_min_slp_008_30_100, 
                                  PSL_30_100, PSL_NC_30_100, case='30-100')

Then we do the Feb2 Mar2 Apr2, its large pertlim experiments

In [ ]:
def plot_o3_evolution_large(var_0008, var_clim, var_F2, var_M2, var_A2, case='30-70'):
    """
    绘制臭氧演变图（大扰动版本）。
    新增参数 var_clim 用于绘制气候态曲线（黑色虚线）。
    对扰动数据仅绘制均值曲线，并以每个时间点的最小/最大值包络区作为阴影区。
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("ticks")
    
    fig, ax = plt.subplots(1, 1, figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    # 绘制基准数据（黑色实线）和气候态（黑色虚线）
    x_base = range(len(var_0008))
    ax.plot(x_base, var_0008, color='black', linewidth=3, label='Base')
    x_clim = range(len(var_clim))
    ax.plot(x_clim, var_clim, color='black', linestyle=':', linewidth=3, label='Clim')
    
    # 绘制 Feb (large pertlim) 数据
    x_F2 = range(31, 31 + len(var_F2.time))
    mean_F2 = var_F2.mean(dim='member')
    env_min_F2 = var_F2.min(dim='member')
    env_max_F2 = var_F2.max(dim='member')
    ax.plot(x_F2, mean_F2, color='navy', linewidth=3, label='Feb, large pertlim')
    ax.fill_between(x_F2, env_min_F2, env_max_F2, color='navy', alpha=0.3)
    
    # 绘制 Mar (large pertlim) 数据
    x_M2 = range(59, 59 + len(var_M2.time))
    mean_M2 = var_M2.mean(dim='member')
    env_min_M2 = var_M2.min(dim='member')
    env_max_M2 = var_M2.max(dim='member')
    ax.plot(x_M2, mean_M2, color='darkred', linewidth=3, label='Mar, large pertlim')
    ax.fill_between(x_M2, env_min_M2, env_max_M2, color='darkred', alpha=0.3)
    
    # 绘制 Apr (large pertlim) 数据
    x_A2 = range(90, 90 + len(var_A2.time))
    mean_A2 = var_A2.mean(dim='member')
    env_min_A2 = var_A2.min(dim='member')
    env_max_A2 = var_A2.max(dim='member')
    ax.plot(x_A2, mean_A2, color='darkorange', linewidth=3, label='Apr, large pertlim')
    ax.fill_between(x_A2, env_min_A2, env_max_A2, color='darkorange', alpha=0.3)
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=22)
    ax.set_ylabel(f'Partial ozone column, {case} hPa (DU)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    plt.legend(fontsize=16)
    
    plt.savefig(f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_0008_large_{case.replace("-","to")}.pdf')
    plt.savefig(f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_0008_large_{case.replace("-","to")}.png')
    
    return fig, ax

# 分别绘制两个压力层范围的图
fig_30_70 = plot_o3_evolution_large(var_0008_30_70_polar,var_clim_30_70_polar, var_F2_30_70, var_M2_30_70, var_A2_30_70, case='30-70')
fig_30_100 = plot_o3_evolution_large(var_0008_30_100_polar, var_clim_30_100_polar, var_F2_30_100, var_M2_30_100, var_A2_30_100, case='30-100')

灰色线（grey）：2008年的基准模拟数据
蓝色线（royalblue）：小扰动实验（Feb, small pertlim）
青色线（teal）：大扰动实验（Feb, large pertlim）
深蓝色线（midnightblue）：Q扰动实验（Feb, Q pert）可能是heating pert热扰动

In [ ]:
def plot_o3_evolution_feb_types(var_0008, var_clim, var_F, var_F2, var_F3, case='30-70'):
    """
    绘制2月份不同类型扰动下的臭氧演化图，添加气候态虚线（var_clim）。
    仅绘制各扰动组的均值曲线，并用每个时间点的最小/最大值包络区（阴影）显示成员分布。
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("ticks")
    
    fig, ax = plt.subplots(1, 1, figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    # 绘制基准数据和气候态
    x_base = range(len(var_0008))
    ax.plot(x_base, var_0008, color='black', linewidth=3, label='Base')
    x_clim = range(len(var_clim))
    ax.plot(x_clim, var_clim, color='black', linestyle=':', linewidth=3, label='Clim')
    
    # Feb, small pertlim
    x_F = range(31, 31 + len(var_F.time))
    mean_F = var_F.mean(dim='member')
    env_min_F = var_F.min(dim='member')
    env_max_F = var_F.max(dim='member')
    ax.plot(x_F, mean_F, color='mediumblue', linewidth=3, label='Feb, small pertlim')
    ax.fill_between(x_F, env_min_F, env_max_F, color='mediumblue', alpha=0.3)
    
    # Feb, large pertlim
    x_F2 = range(31, 31 + len(var_F2.time))
    mean_F2 = var_F2.mean(dim='member')
    env_min_F2 = var_F2.min(dim='member')
    env_max_F2 = var_F2.max(dim='member')
    ax.plot(x_F2, mean_F2, color='forestgreen', linewidth=3, label='Feb, large pertlim')
    ax.fill_between(x_F2, env_min_F2, env_max_F2, color='forestgreen', alpha=0.3)
    
    # Feb, Q pert
    x_F3 = range(31, 31 + len(var_F3.time))
    mean_F3 = var_F3.mean(dim='member')
    env_min_F3 = var_F3.min(dim='member')
    env_max_F3 = var_F3.max(dim='member')
    ax.plot(x_F3, mean_F3, color='purple', linewidth=3, label='Feb, Q pert')
    ax.fill_between(x_F3, env_min_F3, env_max_F3, color='purple', alpha=0.3)
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=22)
    ax.set_ylabel(f'Partial ozone column, {case} hPa (DU)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    plt.legend(fontsize=16)
    
    plt.savefig(f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_0008_Feb_{case.replace("-","to")}.pdf')
    plt.savefig(f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_0008_Feb_{case.replace("-","to")}.png')
    
    return fig, ax

# 分别绘制两个压力层范围的图
fig_30_70 = plot_o3_evolution_feb_types(var_0008_30_70_polar,var_clim_30_70_polar, var_F_30_70, 
                                     var_F2_30_70, var_F3_30_70, case='30-70')
#fig_30_100 = plot_o3_evolution_feb_types(var_0008_30_100_polar,var_clim_30_100_polar, var_F_30_100, 
#                                       var_F2_30_100, var_F3_30_100, case='30-100')

专门看三月不同扰动

In [ ]:
def plot_o3_evolution_mar_types(var_0008, var_clim, var_M, var_M2, var_M3, case='30-70'):
    """
    绘制3月份不同类型扰动下的臭氧演化图，添加气候态虚线（var_clim）。
    仅绘制各扰动组的均值曲线，并用每个时间点的最小/最大值包络区作为阴影。
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("ticks")
    
    fig, ax = plt.subplots(1, 1, figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    # 绘制基准数据和气候态
    x_base = range(len(var_0008))
    ax.plot(x_base, var_0008, color='black', linewidth=3, label='Base')
    x_clim = range(len(var_clim))
    ax.plot(x_clim, var_clim, color='black', linestyle=':', linewidth=3, label='Clim')
    
    # Mar, small pertlim
    x_M = range(59, 59 + len(var_M.time))
    mean_M = var_M.mean(dim='member')
    env_min_M = var_M.min(dim='member')
    env_max_M = var_M.max(dim='member')
    ax.plot(x_M, mean_M, color='crimson', linewidth=3, label='Mar, small pertlim')
    ax.fill_between(x_M, env_min_M, env_max_M, color='crimson', alpha=0.3)
    
    # Mar, large pertlim
    x_M2 = range(59, 59 + len(var_M2.time))
    mean_M2 = var_M2.mean(dim='member')
    env_min_M2 = var_M2.min(dim='member')
    env_max_M2 = var_M2.max(dim='member')
    ax.plot(x_M2, mean_M2, color='indigo', linewidth=3, label='Mar, large pertlim')
    ax.fill_between(x_M2, env_min_M2, env_max_M2, color='indigo', alpha=0.3)
    
    # Mar, heating pert
    x_M3 = range(59, 59 + len(var_M3.time))
    mean_M3 = var_M3.mean(dim='member')
    env_min_M3 = var_M3.min(dim='member')
    env_max_M3 = var_M3.max(dim='member')
    ax.plot(x_M3, mean_M3, color='teal', linewidth=3, label='Mar, heating pert')
    ax.fill_between(x_M3, env_min_M3, env_max_M3, color='teal', alpha=0.3)
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=22)
    ax.set_ylabel(f'Partial ozone column, {case} hPa (DU)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    plt.legend(fontsize=16)
    
    plt.savefig(f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_0008_Mar_{case.replace("-","to")}.pdf')
    plt.savefig(f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_0008_Mar_{case.replace("-","to")}.png')
    
    return fig, ax

# 分别绘制两个压力层范围的图
fig_30_70 = plot_o3_evolution_mar_types(var_0008_30_70_polar,var_clim_30_70_polar, var_M_30_70, 
                                     var_M2_30_70, var_M3_30_70, case='30-70')
#fig_30_100 = plot_o3_evolution_mar_types(var_0008_30_100_polar,var_clim_30_100_polar, var_M_30_100, 
#                                       var_M2_30_100, var_M3_30_100, case='30-100')

下面绘制风的

In [ ]:
def process_waccm_data(file_path, lat=60, plev=1000):
    """处理WACCM数据文件
    
    Args:
        file_path: 数据文件路径
        lat: 指定纬度，默认60°N
        plev: 指定气压层(hPa)，默认1000hPa
    
    Returns:
        处理后的数据变量
    """
    nc_fid = xr.open_mfdataset(file_path, concat_dim='member', combine='nested')
    o3 = nc_fid['U']
    o3 = o3.mean(dim='lon')
    o3 = o3.sel(plev=plev)
    var = o3.sel(lat=lat, method='nearest')
    return var

def process_base_data(file_path, lat=60, plev=1000, months=[1,2,3,4,5,6]):
    """处理基准数据
    
    Args:
        file_path: 基准数据文件路径
        lat: 指定纬度，默认60°N
        plev: 指定气压层(hPa)，默认1000hPa
        months: 需要的月份列表，默认1-6月
    
    Returns:
        基准数据和气候态
    """
    nc_fid = xr.open_dataset(file_path)
    O3 = nc_fid['U']
    O3 = O3.sel(plev=plev)
    O3 = O3.sel(time=nc_fid.time.dt.month.isin(months))
    O3 = O3.mean(dim='lon')
    var = O3.sel(lat=lat, method='nearest')
    
    var_0008 = var.sel(time=var.time.dt.year.isin([8]))
    var_0008_clim = var.groupby("time.dayofyear").mean()
    
    return var_0008, var_0008_clim




In [ ]:

def plot_waccm_results(var_0008, var_0008_clim, var_J, var_F, var_M, save_path, lat=60, plev=1000):
    """
    绘制WACCM结果图。
    - 基准数据(var_0008)和气候态(var_0008_clim)以黑色分别作为实线和虚线绘制。
    - 各月份数据（Jan、Feb、Mar）仅绘制均值曲线，并用每个时间点的最小到最大值包络区作为阴影显示。
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("ticks")
    
    fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    # 基准和气候态（黑色）
    x_base = range(len(var_0008))
    ax.plot(x_base, var_0008, color='black', linewidth=3, label='Base')
    x_clim = range(len(var_0008_clim))
    ax.plot(x_clim, var_0008_clim, color='black', linestyle=':', linewidth=3, label='Clim')
    
    # Jan数据
    x_J = range(len(var_J.time))
    mean_J = var_J.mean(dim='member')
    env_min_J = var_J.min(dim='member')
    env_max_J = var_J.max(dim='member')
    ax.plot(x_J, mean_J, color='forestgreen', linewidth=3, label='Jan small pertlim')
    ax.fill_between(x_J, env_min_J, env_max_J, color='forestgreen', alpha=0.3)
    
    # Feb数据：从31开始
    x_F = range(31, 31 + len(var_F.time))
    mean_F = var_F.mean(dim='member')
    env_min_F = var_F.min(dim='member')
    env_max_F = var_F.max(dim='member')
    ax.plot(x_F, mean_F, color='royalblue', linewidth=3, label='Feb small pertlim')
    ax.fill_between(x_F, env_min_F, env_max_F, color='royalblue', alpha=0.3)
    
    # Mar数据：从59开始
    x_M = range(59, 59 + len(var_M.time))
    mean_M = var_M.mean(dim='member')
    env_min_M = var_M.min(dim='member')
    env_max_M = var_M.max(dim='member')
    ax.plot(x_M, mean_M, color='hotpink', linewidth=3, label='Mar small pertlim')
    ax.fill_between(x_M, env_min_M, env_max_M, color='hotpink', alpha=0.3)
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=22)
    
    hpa = plev/100
    lat_label = 'N' if lat >= 0 else 'S'
    ax.set_ylabel(f'Zonal mean zonal wind, {abs(lat)}°{lat_label}, {hpa:.0f} hPa (m/s)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    
    plt.legend(fontsize=16)
    plt.savefig(f'{save_path}.pdf')
    plt.savefig(f'{save_path}.png')
    
    return fig, ax

# 3. 绘图
lat = 60  # 60°N
plev = 1000  # 1000 Pa = 10 hPa
var_0008, var_0008_clim = process_base_data(
    file_path='/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc',
    lat=lat,
    plev=plev
)
jan_data = process_waccm_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc', lat=lat, plev=plev)
feb_data = process_waccm_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc', lat=lat, plev=plev)
mar_data = process_waccm_data('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc', lat=lat, plev=plev)

save_path = '/home/weiji/restart_exam/plots/U_evolution_min_restart_0008'
plot_waccm_results(var_0008, var_0008_clim, jan_data, feb_data, mar_data, save_path, lat=lat, plev=plev)

In [ ]:
def plot_february_perturbations(var_0008, var_0008_clim, var_F, var_F2, var_F3, var_FNC, save_path, lat=60, plev=1000):
    """
    绘制WACCM结果图（风数据版）。
    - 基准数据(var_0008)和气候态(var_0008_clim)分别以黑色实线和虚线绘制。
    - 四个二月扰动数据（var_F, var_F2, var_F3, var_FNC）仅绘制均值曲线，
      并用每个时间点的最小值到最大值的包络区作为阴影显示。
      
    参数：
      var_0008, var_0008_clim : 基准和气候态数据
      var_F, var_F2, var_F3, var_FNC : 四个二月扰动实验数据
      save_path              : 保存图像的路径（不含扩展名）
      lat                    : 绘图时所用的纬度（正数表示北纬，负数表示南纬）
      plev                   : 所用气压层（单位：hPa）
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("ticks")
    
    fig, ax = plt.subplots(1, 1, figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    # 绘制基准数据和气候态数据（黑色实线和虚线）
    x_base = range(len(var_0008))
    ax.plot(x_base, var_0008, color='black', linewidth=3, label='Base')
    x_clim = range(len(var_0008_clim))
    ax.plot(x_clim, var_0008_clim, color='black', linestyle=':', linewidth=3, label='Clim')
    
    # 绘制四个二月扰动数据（均从 x=31 开始）
    # Feb 数据
    x_F = range(31, 31 + len(var_F.time))
    mean_F = var_F.mean(dim='member')
    env_min_F = var_F.min(dim='member')
    env_max_F = var_F.max(dim='member')
    ax.plot(x_F, mean_F, color='royalblue', linewidth=3, label='Feb small pertlim')
    ax.fill_between(x_F, env_min_F, env_max_F, color='royalblue', alpha=0.3)
    
    # Feb 2 数据
    x_F2 = range(31, 31 + len(var_F2.time))
    mean_F2 = var_F2.mean(dim='member')
    env_min_F2 = var_F2.min(dim='member')
    env_max_F2 = var_F2.max(dim='member')
    ax.plot(x_F2, mean_F2, color='mediumseagreen', linewidth=3, label='Feb large pertlim')
    ax.fill_between(x_F2, env_min_F2, env_max_F2, color='mediumseagreen', alpha=0.3)
    
    # Feb 3 数据
    x_F3 = range(31, 31 + len(var_F3.time))
    mean_F3 = var_F3.mean(dim='member')
    env_min_F3 = var_F3.min(dim='member')
    env_max_F3 = var_F3.max(dim='member')
    ax.plot(x_F3, mean_F3, color='darkorange', linewidth=3, label='Feb Q pertlim')
    ax.fill_between(x_F3, env_min_F3, env_max_F3, color='darkorange', alpha=0.3)
    
    # Feb NOCOUPL 数据
    x_FNC = range(31, 31 + len(var_FNC.time))
    mean_FNC = var_FNC.mean(dim='member')
    env_min_FNC = var_FNC.min(dim='member')
    env_max_FNC = var_FNC.max(dim='member')
    ax.plot(x_FNC, mean_FNC, color='slategray', linewidth=3, label='Feb NOCOUPL')
    ax.fill_between(x_FNC, env_min_FNC, env_max_FNC, color='slategray', alpha=0.3)
    
    # 设置 x 轴范围和刻度
    ax.set_xlim(0, 150)
    ax.set_xticks([0, 31, 59, 89, 120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun'], fontsize=22)
    
    # 设置 y 轴标签（风速单位 m/s），并根据 plev 转换 hPa 显示
    hpa = plev / 100.0
    lat_label = 'N' if lat >= 0 else 'S'
    ax.set_ylabel(f'Zonal mean zonal wind, {abs(lat)}°{lat_label}, {hpa:.0f} hPa (m/s)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    
    plt.legend(fontsize=16)
    plt.savefig(f'{save_path}.pdf', bbox_inches="tight")
    plt.savefig(f'{save_path}.png', bbox_inches="tight")
    
    return fig, ax


In [ ]:

# 处理其他扰动实验数据

var_F2 = process_waccm_data(
    '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc',
    lat=lat, plev=plev
)
var_F3 = process_waccm_data(
    '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc',
    lat=lat, plev=plev
)
var_FNC = process_waccm_data(
    '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/BWCN.e122.f19_g16.002.0008-02.*.nc',
    lat=lat, plev=plev
)

# 绘制二月份扰动实验对比图
save_path = '/home/weiji/restart_exam/plots/U_evolution_min_restart_0008_Feb'
# 单行调用示例
plot_february_perturbations(var_0008, var_0008_clim, feb_data, var_F2, var_F3, var_FNC, save_path, lat=lat, plev=plev)

三月份扰动实验的绘图

In [ ]:
def plot_march_perturbations(var_0008, var_0008_clim, var_M, var_M2, var_M3, save_path, lat=60, plev=1000):
    """
    绘制WACCM结果图（风数据版）。
    - 基准数据 (var_0008) 和气候态 (var_0008_clim) 分别以黑色实线和虚线绘制。
    - 三种不同扰动实验数据（均为三月数据）仅绘制均值曲线，并用每个时间点的最小到最大值包络区作为阴影显示。
    
    参数：
      var_0008, var_0008_clim : 基准和气候态数据
      var_M, var_M2, var_M3   : 三种扰动实验数据（例如：small pert, large pert, Q pert）
      save_path              : 保存图像的路径（不含扩展名）
      lat                    : 绘图时所用的纬度（正数表示北纬，负数表示南纬）
      plev                   : 所用气压层（单位：hPa）
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("ticks")
    
    fig, ax = plt.subplots(1, 1, figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    # 绘制基准和气候态（黑色实线和虚线）
    x_base = range(len(var_0008))
    ax.plot(x_base, var_0008, color='black', linewidth=3, label='Base')
    x_clim = range(len(var_0008_clim))
    ax.plot(x_clim, var_0008_clim, color='black', linestyle=':', linewidth=3, label='Clim')
    
    # 绘制三种扰动实验数据（均从三月开始，x 坐标从 59 开始）
    # Mar, small pert
    x_M = range(59, 59 + len(var_M.time))
    mean_M = var_M.mean(dim='member')
    env_min_M = var_M.min(dim='member')
    env_max_M = var_M.max(dim='member')
    ax.plot(x_M, mean_M, color='crimson', linewidth=3, label='Mar, small pert')
    ax.fill_between(x_M, env_min_M, env_max_M, color='crimson', alpha=0.3)
    
    # Mar, large pert
    x_M2 = range(59, 59 + len(var_M2.time))
    mean_M2 = var_M2.mean(dim='member')
    env_min_M2 = var_M2.min(dim='member')
    env_max_M2 = var_M2.max(dim='member')
    ax.plot(x_M2, mean_M2, color='indigo', linewidth=3, label='Mar, large pert')
    ax.fill_between(x_M2, env_min_M2, env_max_M2, color='indigo', alpha=0.3)
    
    # Mar, Q pert
    x_M3 = range(59, 59 + len(var_M3.time))
    mean_M3 = var_M3.mean(dim='member')
    env_min_M3 = var_M3.min(dim='member')
    env_max_M3 = var_M3.max(dim='member')
    ax.plot(x_M3, mean_M3, color='teal', linewidth=3, label='Mar, Q pert')
    ax.fill_between(x_M3, env_min_M3, env_max_M3, color='teal', alpha=0.3)
    
    # 设置 x 轴刻度及标签
    ax.set_xlim(0, 150)
    ax.set_xticks([0, 31, 59, 89, 120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun'], fontsize=22)
    
    # 设置 y 轴标签：风速数据（单位 m/s），气压层转换为 hPa 显示
    hpa = plev / 100.0
    lat_label = 'N' if lat >= 0 else 'S'
    ax.set_ylabel(f'Zonal mean zonal wind, {abs(lat)}°{lat_label}, {hpa:.0f} hPa (m/s)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    
    plt.legend(fontsize=16)
    
    # 同时保存为 PDF 和 PNG
    plt.savefig(f'{save_path}.pdf', bbox_inches="tight")
    plt.savefig(f'{save_path}.png', bbox_inches="tight")
    
    return fig, ax

In [ ]:
# 处理其他扰动实验数据
var_M2 = process_waccm_data(
    '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc',
    lat=lat, plev=plev
)
var_M3 = process_waccm_data(
    '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc',
    lat=lat, plev=plev
)

# 绘制三月份扰动实验对比图
save_path = '/home/weiji/restart_exam/plots/U_evolution_min_restart_0008_Mar'
plot_march_perturbations(var_0008,var_0008_clim, mar_data, var_M2, var_M3, save_path, lat=lat, plev=plev)

下面是温度处理，首先读取数据

In [ ]:
def process_temp_data(file_path, plev=5000, lat_range=(60, 90)):
    """处理WACCM温度数据文件
    
    Args:
        file_path: 数据文件路径
        plev: 气压层(Pa)，默认5000Pa
        lat_range: 纬度范围元组(min_lat, max_lat)，默认(60,90)
    
    Returns:
        处理后的数据变量
    """
    nc_fid = xr.open_mfdataset(file_path, concat_dim='member', combine='nested')
    temp = nc_fid['T']
    temp = temp.mean(dim='lon')
    temp = temp.sel(plev=plev)
    var = temp.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')
    return var

def process_temp_base_data(file_path, plev=5000, lat_range=(60, 90), months=[1,2,3,4,5,6]):
    """处理温度基准数据
    
    Args:
        file_path: 基准数据文件路径
        plev: 气压层(Pa)，默认5000Pa
        lat_range: 纬度范围元组(min_lat, max_lat)，默认(60,90)
        months: 需要的月份列表，默认1-6月
    
    Returns:
        基准数据和气候态
    """
    nc_fid = xr.open_dataset(file_path)
    temp = nc_fid['T']
    temp = temp.sel(plev=plev)
    temp = temp.sel(time=nc_fid.time.dt.month.isin(months))
    temp = temp.mean(dim='lon')
    var = temp.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')
    
    var_0008 = var.sel(time=var.time.dt.year.isin([8]))
    var_0008_clim = var.groupby("time.dayofyear").mean()
    
    return var_0008, var_0008_clim

In [ ]:
# 设置基本参数
plev = 5000  # 50 hPa
lat_range = (60, 90)  # 60°N-90°N
base_dir = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart'

# 处理基准数据
var_0008, var_0008_clim = process_temp_base_data(
    f'{base_dir}/BWCN.e122.f19_g16.merged.nc',
    plev=plev,
    lat_range=lat_range
)

# 处理各月份数据
jan_data = process_temp_data(
    f'{base_dir}/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc',
    plev=plev,
    lat_range=lat_range
)

feb_data = process_temp_data(
    f'{base_dir}/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc',
    plev=plev,
    lat_range=lat_range
)

# 二月份其他扰动实验
feb_data_2 = process_temp_data(
    f'{base_dir}/BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc',
    plev=plev,
    lat_range=lat_range
)

feb_data_3 = process_temp_data(
    f'{base_dir}/BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc',
    plev=plev,
    lat_range=lat_range
)

feb_data_nc = process_temp_data(
    f'{base_dir}/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/BWCN.e122.f19_g16.002.0008-02.*.nc',
    plev=plev,
    lat_range=lat_range
)

# 三月份数据
mar_data = process_temp_data(
    f'{base_dir}/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc',
    plev=plev,
    lat_range=lat_range
)

mar_data_2 = process_temp_data(
    f'{base_dir}/BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc',
    plev=plev,
    lat_range=lat_range
)

mar_data_3 = process_temp_data(
    f'{base_dir}/BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc',
    plev=plev,
    lat_range=lat_range
)

极地最低温度的时间演变
 蓝色虚线：标记了197K的氯活化阈值，这是一个重要的化学过程阈值

In [ ]:
def plot_temp_results(var_0008, var_0008_clim, var_J, var_F, var_M, save_path, plev=5000, lat_range=(60, 90)):
    """
    绘制WACCM温度结果图。
    - 基准数据(var_0008)和气候态(var_0008_clim)均以黑色绘制（实线和虚线）。
    - Jan、Feb、Mar数据仅绘制均值曲线，并以每个时间点的最小值到最大值包络区作为阴影显示。
    - 同时添加氯活化阈值线（197K）。
    """
    sns.set_style("ticks")
    
    fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    # 基准与气候态（黑色）
    x_base = range(len(var_0008))
    ax.plot(x_base, var_0008, color='black', linewidth=3, label='Base')
    x_clim = range(len(var_0008_clim))
    ax.plot(x_clim, var_0008_clim, color='black', linestyle=':', linewidth=3, label='Clim')
    
    # Jan数据
    x_J = range(len(var_J.time))
    mean_J = var_J.mean(dim='member')
    env_min_J = var_J.min(dim='member')
    env_max_J = var_J.max(dim='member')
    ax.plot(x_J, mean_J, color='forestgreen', linewidth=3, label='Jan small pertlim')
    ax.fill_between(x_J, env_min_J, env_max_J, color='forestgreen', alpha=0.3)
    
    # Feb数据：从31开始
    x_F = range(31, 31 + len(var_F.time))
    mean_F = var_F.mean(dim='member')
    env_min_F = var_F.min(dim='member')
    env_max_F = var_F.max(dim='member')
    ax.plot(x_F, mean_F, color='royalblue', linewidth=3, label='Feb small pertlim')
    ax.fill_between(x_F, env_min_F, env_max_F, color='royalblue', alpha=0.3)
    
    # Mar数据：从59开始
    x_M = range(59, 59 + len(var_M.time))
    mean_M = var_M.mean(dim='member')
    env_min_M = var_M.min(dim='member')
    env_max_M = var_M.max(dim='member')
    ax.plot(x_M, mean_M, color='hotpink', linewidth=3, label='Mar small pertlim')
    ax.fill_between(x_M, env_min_M, env_max_M, color='hotpink', alpha=0.3)
    
    # 添加氯活化阈值线
    ax.axhline(y=197, color='royalblue', linestyle='--')
    ax.text(5, 194, 'Cl activation threshold', horizontalalignment='left', fontsize=20, color='royalblue')
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=22)
    
    hpa = plev/100
    ax.set_ylabel(f'Minimum Temperature, {lat_range[0]}-{lat_range[1]}°N, {hpa:.0f} hPa (K)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    
    plt.legend(fontsize=16)
    plt.savefig(f'{save_path}.pdf')
    plt.savefig(f'{save_path}.png')
    
    return fig, ax

In [ ]:
# 蓝色虚线：标记了197K的氯活化阈值，这是一个重要的化学过程阈值
save_path = '/home/weiji/restart_exam/plots/T_min_evolution_min_restart_0008'
plot_temp_results(
    var_0008, var_0008_clim,
    jan_data, feb_data, mar_data,
    save_path,
    plev=plev,
    lat_range=lat_range
)

二月份温度扰动实验的绘图

In [ ]:
def plot_temp_february_perturbations(var_0008, var_0008_clim, var_F, var_F2, var_F3, var_FNC, save_path, plev=5000, lat_range=(60, 90)):
    """
    绘制二月份不同扰动实验的温度结果对比图，
    添加基准温度（var_0008）和气候态温度（var_0008_clim）曲线（均为黑色），
    其他实验数据仅显示均值曲线，并以每个时间点的最小/最大值包络区作为阴影区。
    """
    sns.set_style("ticks")
    
    fig, ax = plt.subplots(1,1,figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    # 绘制基准与气候态
    x_base = range(len(var_0008))
    ax.plot(x_base, var_0008, color='black', linewidth=3, label='Base')
    x_clim = range(len(var_0008_clim))
    ax.plot(x_clim, var_0008_clim, color='black', linestyle=':', linewidth=3, label='Clim')
    
    # Feb, small pertlim
    x_F = range(31, 31 + len(var_F.time))
    mean_F = var_F.mean(dim='member')
    env_min_F = var_F.min(dim='member')
    env_max_F = var_F.max(dim='member')
    ax.plot(x_F, mean_F, color='navy', linewidth=3, label='Feb, small pertlim')
    ax.fill_between(x_F, env_min_F, env_max_F, color='navy', alpha=0.3)
    
    # Feb, large pertlim
    x_F2 = range(31, 31 + len(var_F2.time))
    mean_F2 = var_F2.mean(dim='member')
    env_min_F2 = var_F2.min(dim='member')
    env_max_F2 = var_F2.max(dim='member')
    ax.plot(x_F2, mean_F2, color='forestgreen', linewidth=3, label='Feb, large pertlim')
    ax.fill_between(x_F2, env_min_F2, env_max_F2, color='forestgreen', alpha=0.3)
    
    # Feb, Q pert
    x_F3 = range(31, 31 + len(var_F3.time))
    mean_F3 = var_F3.mean(dim='member')
    env_min_F3 = var_F3.min(dim='member')
    env_max_F3 = var_F3.max(dim='member')
    ax.plot(x_F3, mean_F3, color='darkorange', linewidth=3, label='Feb, Q pert')
    ax.fill_between(x_F3, env_min_F3, env_max_F3, color='darkorange', alpha=0.3)
    
    # Feb, NOCOUPL
    x_FNC = range(31, 31 + len(var_FNC.time))
    mean_FNC = var_FNC.mean(dim='member')
    env_min_FNC = var_FNC.min(dim='member')
    env_max_FNC = var_FNC.max(dim='member')
    ax.plot(x_FNC, mean_FNC, color='darkred', linewidth=3, label='Feb, NOCOUPL')
    ax.fill_between(x_FNC, env_min_FNC, env_max_FNC, color='darkred', alpha=0.3)
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=22)
    ax.set_ylabel(f'Minimum Temperature, {lat_range[0]}-{lat_range[1]}°N, {plev/100:.0f} hPa (K)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    plt.legend(fontsize=16)
    
    plt.savefig(f'{save_path}.pdf')
    plt.savefig(f'{save_path}.png')
    
    return fig, ax

In [ ]:
# 使用之前处理好的数据绘图
save_path = '/home/weiji/restart_exam/plots/T_min_evolution_min_restart_0008_Feb'
plot_temp_february_perturbations(
    var_0008,var_0008_clim, feb_data, feb_data_2, feb_data_3, feb_data_nc,
    save_path,
    plev=plev,
    lat_range=lat_range
)

三月份温度扰动实验的绘图

In [ ]:
def plot_temp_march_perturbations(var_0008, var_0008_clim, var_M, var_M2, var_M3, save_path, plev=5000, lat_range=(60, 90)):
    """
    绘制三月份不同扰动实验的温度结果对比图，
    添加基准温度（var_0008）与气候态温度（var_0008_clim）曲线（黑色），
    其他实验数据仅显示均值曲线，并以每个时间点的最小/最大值包络区作为阴影区。
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("ticks")
    
    fig, ax = plt.subplots(1, 1, figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    # 绘制基准温度与气候态温度（黑色）
    x_base = range(len(var_0008))
    ax.plot(x_base, var_0008, color='black', linewidth=3, label='Base')
    x_clim = range(len(var_0008_clim))
    ax.plot(x_clim, var_0008_clim, color='black', linestyle=':', linewidth=3, label='Clim')
    
    # Mar, small pertlim：从59开始
    x_M = range(59, 59 + len(var_M.time))
    mean_M = var_M.mean(dim='member')
    env_min_M = var_M.min(dim='member')
    env_max_M = var_M.max(dim='member')
    ax.plot(x_M, mean_M, color='crimson', linewidth=3, label='Mar, small pertlim')
    ax.fill_between(x_M, env_min_M, env_max_M, color='crimson', alpha=0.3)
    
    # Mar, large pertlim
    x_M2 = range(59, 59 + len(var_M2.time))
    mean_M2 = var_M2.mean(dim='member')
    env_min_M2 = var_M2.min(dim='member')
    env_max_M2 = var_M2.max(dim='member')
    ax.plot(x_M2, mean_M2, color='indigo', linewidth=3, label='Mar, large pertlim')
    ax.fill_between(x_M2, env_min_M2, env_max_M2, color='indigo', alpha=0.3)
    
    # Mar, Q pert
    x_M3 = range(59, 59 + len(var_M3.time))
    mean_M3 = var_M3.mean(dim='member')
    env_min_M3 = var_M3.min(dim='member')
    env_max_M3 = var_M3.max(dim='member')
    ax.plot(x_M3, mean_M3, color='teal', linewidth=3, label='Mar, Q pert')
    ax.fill_between(x_M3, env_min_M3, env_max_M3, color='teal', alpha=0.3)
    
    # 设置x轴刻度与标签
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=22)
    
    # 设置y轴标签
    ax.set_ylabel(f'Minimum Temperature, {lat_range[0]}-{lat_range[1]}°N, {plev/100:.0f} hPa (K)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    
    plt.legend(fontsize=16)
    plt.savefig(f'{save_path}.pdf', bbox_inches='tight')
    plt.savefig(f'{save_path}.png', bbox_inches='tight')
    
    return fig, ax

In [ ]:
# 使用之前处理好的数据绘图
save_path = '/home/weiji/restart_exam/plots/T_min_evolution_min_restart_0008_Mar'
# 调用绘图子程序
plot_temp_march_perturbations(var_0008, var_0008_clim, mar_data, mar_data_2, mar_data_3, save_path, plev=plev, lat_range=lat_range)

接下来拓展到另外四年

In [ ]:
############################
# Extended Code for New Cases:
# Plot Ozone, Temperature and Wind Evolution using case‐specific reference lines.
# the reference lines are extracted from the same dataset by selecting the year equal to the case number.
############################

def process_o3_data_newcase(file_pattern, var_prefix):
    """
    读取给定 file_pattern 内的多个 nc 文件，
    计算 O3 的局部平均（先取经向平均，再选取 60°–90°，加权平均）；
    分别计算 30–70 和 30–100 hPa 的部分臭氧柱。
    
    参数：
      file_pattern: 文件匹配模式（包含通配符*）
      var_prefix: 用于返回字典中变量命名的前缀（例如 "F" 或 "M"）
      
    返回一个字典，键包括：
      "{var_prefix}_30_70" 和 "{var_prefix}_30_100"
    """
    # 读取数据（ensemble成员沿 member 维度拼接）
    nc_fid = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    # 先经向平均
    o3 = nc_fid['O3'].mean(dim='lon')
    # 计算 30–70 hPa 部分臭氧柱
    o3_30_70 = calc_parc_o3(o3, 30, 70)
    var_30_70 = o3_30_70.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(var_30_70.lat))
    var_30_70 = var_30_70.weighted(weights).mean(dim='lat')
    # 计算 30–100 hPa 部分臭氧柱
    o3_30_100 = calc_parc_o3(o3, 30, 100)
    var_30_100 = o3_30_100.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(var_30_100.lat))
    var_30_100 = var_30_100.weighted(weights).mean(dim='lat')
    return {f'{var_prefix}_30_70': var_30_70, f'{var_prefix}_30_100': var_30_100}





############################
# Extended code for new cases: Plot Ozone, Temperature and Wind evolution
# using case‐specific reference lines (for years 13, 14, and 19 respectively)
############################

In [ ]:
############################
# Extended Code for New Cases:
# Plot Ozone, Temperature and Wind Evolution using case‐specific reference lines.
# For each new case (e.g. '0013', '0014', '0019'),
# the reference lines are extracted from the same dataset by selecting the year equal to the case number.
############################



##########################################
# 1. Ozone Evolution Plot with Case‐Specific Reference
##########################################

def plot_o3_evolution_newcase_with_ref(var_F, var_M, var_clim, pressure_range='30-70', case_id=''):
    """
    绘制新案例的臭氧演化图（仅Feb和Mar），在图中叠加参考曲线（根据 case_id 提取）。
    基准曲线与气候态曲线均为黑色，其他数据仅绘制均值曲线，并以每个时间点的最小/最大值包络区作为阴影区。
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("ticks")
    
    fig, ax = plt.subplots(1, 1, figsize=(12,8), constrained_layout=True)
    
    # 绘制新案例 Feb 数据
    if var_F is not None:
        x_F = range(31, 31 + len(var_F.time))
        mean_F = var_F.mean(dim='member')
        env_min_F = var_F.min(dim='member')
        env_max_F = var_F.max(dim='member')
        ax.plot(x_F, mean_F, color='navy', linewidth=3, label='Feb small pertlim')
        ax.fill_between(x_F, env_min_F, env_max_F, color='navy', alpha=0.3)
            
    # 绘制新案例 Mar 数据
    if var_M is not None:
        x_M = range(59, 59 + len(var_M.time))
        mean_M = var_M.mean(dim='member')
        env_min_M = var_M.min(dim='member')
        env_max_M = var_M.max(dim='member')
        ax.plot(x_M, mean_M, color='crimson', linewidth=3, label='Mar small pertlim')
        ax.fill_between(x_M, env_min_M, env_max_M, color='crimson', alpha=0.3)
    
    # 提取参考曲线：根据 case_id 从 var_clim 中选取对应年份的曲线
    try:
        case_year = int(case_id)
    except:
        case_year = None
        
    ref_curve = None
    ref_clim = None
    if case_year is not None:
        temp_select = var_clim.sel(time=var_clim.time.dt.month.isin([1,2,3,4,5,6]))
        ref_curve = temp_select.sel(time=temp_select.time.dt.year==case_year)
        ref_clim = temp_select.groupby("time.dayofyear").mean()
    
    if ref_curve is not None:
        x_ref = range(len(ref_curve))
        ax.plot(x_ref, ref_curve, color='black', linestyle='--', linewidth=2, label=f'Ref ({case_year})')
    if ref_clim is not None:
        x_ref_clim = range(len(ref_clim))
        ax.plot(x_ref_clim, ref_clim, color='black', linestyle=':', linewidth=2, label='Clim (all)')
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=22)
    ax.set_ylabel(f'Partial ozone column ({pressure_range} hPa) (DU)', fontsize=22)
    plt.legend(fontsize=16)
    
    save_pdf = f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_{case_id}_{pressure_range.replace("-","to")}_withRef.pdf'
    save_png = f'/home/weiji/restart_exam/plots/O3_evolution_min_restart_{case_id}_{pressure_range.replace("-","to")}_withRef.png'
    plt.savefig(save_pdf)
    plt.savefig(save_png)
    plt.close(fig)
    print(f"Saved ozone evolution plot with reference for case {case_id} ({pressure_range})")
    return fig, ax

##########################################
# Loop over new cases: '0013', '0014', '0019', '0003'
##########################################

new_cases = ['0013', '0014', '0019', '0003']

for case in new_cases:
    base_path = f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{case}"
    
    #### Ozone Data Processing ####
    case_data = {}  # dictionary to store ozone data for Feb and Mar for this case
    feb_dir = os.path.join(base_path, "Feb")
    mar_dir = os.path.join(base_path, "Mar")
    if os.path.isdir(feb_dir):
        feb_pattern = os.path.join(feb_dir, f"BWCN.e122.f19_g16.002.{case}-02.*.nc")
        try:
            data_F = process_o3_data_newcase(feb_pattern, var_prefix="F")
            case_data['F_30_70'] = data_F["F_30_70"]
            case_data['F_30_100'] = data_F["F_30_100"]
        except Exception as e:
            print(f"Error processing ozone Feb data for case {case}: {e}")
            case_data['F_30_70'] = None
            case_data['F_30_100'] = None
    else:
        print(f"Feb directory not found for case {case} (ozone)")
        case_data['F_30_70'] = None
        case_data['F_30_100'] = None
    if os.path.isdir(mar_dir):
        mar_pattern = os.path.join(mar_dir, f"BWCN.e122.f19_g16.002.{case}-03.*.nc")
        try:
            data_M = process_o3_data_newcase(mar_pattern, var_prefix="M")
            case_data['M_30_70'] = data_M["M_30_70"]
            case_data['M_30_100'] = data_M["M_30_100"]
        except Exception as e:
            print(f"Error processing ozone Mar data for case {case}: {e}")
            case_data['M_30_70'] = None
            case_data['M_30_100'] = None
    else:
        print(f"Mar directory not found for case {case} (ozone)")
        case_data['M_30_70'] = None
        case_data['M_30_100'] = None
    
    # Plot ozone evolution for both pressure ranges
    for pressure in ['30-70', '30-100']:
        key_F = 'F_30_70' if pressure=='30-70' else 'F_30_100'
        var_climcc = var_30_70_polar if pressure=='30-70' else var_30_100_polar
        key_M = 'M_30_70' if pressure=='30-70' else 'M_30_100'
        var_F = case_data.get(key_F, None)
        var_M = case_data.get(key_M, None)
        if (var_F is None) and (var_M is None):
            print(f"No ozone data for pressure {pressure} in case {case} – skipping ozone plot.")
        else:
            plot_o3_evolution_newcase_with_ref(var_F, var_M,var_climcc, pressure_range=pressure, case_id=case)
    
    

In [ ]:
##############################
# Additional Code for Extra Cases: Temperature and Wind Evolution with Case‐Specific Reference
##############################

import os
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

def process_temp_data(file_path, plev=5000, lat_range=(60, 90)):
    """处理WACCM温度数据文件
    
    Args:
        file_path: 数据文件路径（可以是通配符）
        plev: 氣压层(Pa)，默认5000Pa
        lat_range: 纬度范围元组(min_lat, max_lat)，默认(60,90)
    
    Returns:
        var: (time, member) 维度的温度数据（已在 lon 上做平均，并在 lat 范围内取 min）
    """
    nc_fid = xr.open_mfdataset(file_path, concat_dim='member', combine='nested')
    temp = nc_fid['T']
    # zonal mean
    temp = temp.mean(dim='lon')
    # 选定气压层
    temp = temp.sel(plev=plev)
    # 在纬度范围内取最小值
    var = temp.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')
    return var

def process_temp_base_data(file_path, plev=5000, lat_range=(60, 90), months=[1,2,3,4,5,6]):
    """处理温度基准数据
    
    Args:
        file_path: 基准数据文件路径（merged.nc）
        plev: 氣压层(Pa)，默认5000Pa
        lat_range: 纬度范围元组(min_lat, max_lat)，默认(60,90)
        months: 需要的月份列表，默认[1,2,3,4,5,6]
    
    Returns:
        var: 包含指定月份(1~6月)的全部年份的温度数据（已做 lon 平均 & lat 范围内 min）
        var_clim: 上述 var 的日气候态 (dayofyear 平均)
    """
    nc_fid = xr.open_dataset(file_path)
    temp = nc_fid['T']
    # 选定气压层
    temp = temp.sel(plev=plev)
    # 筛选指定月份
    temp = temp.sel(time=temp.time.dt.month.isin(months))
    # zonal mean
    temp = temp.mean(dim='lon')
    # 在纬度范围内取最小值
    var = temp.sel(lat=slice(lat_range[0], lat_range[1])).min(dim='lat')
    
    # 计算日气候态（对所有年份做 dayofyear 分组求平均）
    var_clim = var.groupby("time.dayofyear").mean()
    
    return var, var_clim

def process_waccm_data(file_path, lat=60, plev=1000):
    """处理WACCM风数据文件
    
    Args:
        file_path: 数据文件路径（可以是通配符）
        lat: 指定纬度，默认60°N
        plev: 指定气压层(Pa)，默认1000Pa (即10 hPa)
    
    Returns:
        var: (time, member) 维度的风数据（已在 lon 上做平均，并在给定 lat 取最近网格）
    """
    nc_fid = xr.open_mfdataset(file_path, concat_dim='member', combine='nested')
    o3 = nc_fid['U']
    # zonal mean
    o3 = o3.mean(dim='lon')
    # 选定气压层
    o3 = o3.sel(plev=plev)
    # 在给定纬度插值/选点
    var = o3.sel(lat=lat, method='nearest')
    return var

def process_base_data(file_path, lat=60, plev=1000, months=[1,2,3,4,5,6]):
    """处理风场基准数据
    
    Args:
        file_path: 基准数据文件路径（merged.nc）
        lat: 指定纬度，默认60°N
        plev: 指定气压层(Pa)，默认1000Pa (即10 hPa)
        months: 需要的月份列表，默认[1,2,3,4,5,6]
    
    Returns:
        var: 包含指定月份(1~6月)的全部年份的风数据（已做 lon 平均 & lat=60°N）
        var_clim: 上述 var 的日气候态 (dayofyear 平均)
    """
    nc_fid = xr.open_dataset(file_path)
    O3 = nc_fid['U']
    # 选定气压层
    O3 = O3.sel(plev=plev)
    # 筛选指定月份
    O3 = O3.sel(time=O3.time.dt.month.isin(months))
    # zonal mean
    O3 = O3.mean(dim='lon')
    # 在给定纬度插值/选点
    var = O3.sel(lat=lat, method='nearest')
    
    # 计算日气候态（对所有年份做 dayofyear 分组求平均）
    var_clim = var.groupby("time.dayofyear").mean()
    
    return var, var_clim

def plot_temp_evolution_newcase_with_ref(var_base, var_base_clim, var_F, var_M, case_id, save_path, plev=5000, lat_range=(60, 90)):
    """
    绘制额外case的温度演变图（仅Feb和Mar），并叠加case特定的参考曲线。
    参数：
      var_base, var_base_clim: 来自merged.nc文件的温度基准数据和按日气候态
      var_F, var_M: 对应额外case中Feb和Mar的温度数据（由 process_temp_data 获得）
      case_id: 字符串形式的case编号（例如 '0013'）
      save_path: 保存图形时的文件名前缀
      plev: 氣压层（Pa），默认5000（即50 hPa）
      lat_range: 纬度范围，默认 (60,90)
    """
    sns.set_style("ticks")
    fig, ax = plt.subplots(1, 1, figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    try:
        case_year = int(case_id)
    except Exception as e:
        case_year = None

    ref_curve = None
    ref_clim = None
    if case_year is not None and var_base is not None:
        # 从 var_base 中再次按年份筛选出该 case_year
        ref_curve = var_base.sel(time=var_base.time.dt.year.isin([case_year]))
        # 直接使用传进来的 var_base_clim 作为气候态
        ref_clim = var_base_clim
    
    # 绘制 base (case_year) 曲线
    if ref_curve is not None and len(ref_curve.time) > 0:
        x_ref = range(len(ref_curve.time))
        ax.plot(x_ref, ref_curve, color='black', linewidth=3, label=f'Base ({case_year})')
    
    # 绘制气候态
    if ref_clim is not None and len(ref_clim.dayofyear) > 0:
        x_ref_clim = range(len(ref_clim.dayofyear))
        ax.plot(x_ref_clim, ref_clim, color='black', linestyle=':', linewidth=3, label='Clim (all)')
    
    # 绘制额外case的Feb数据（从x=31开始）
    if var_F is not None:
        x_F = range(31, 31 + len(var_F.time))
        mean_F = var_F.mean(dim='member')
        env_min_F = var_F.min(dim='member')
        env_max_F = var_F.max(dim='member')
        ax.plot(x_F, mean_F, color='royalblue', linewidth=3, label='Feb small pertlim')
        ax.fill_between(x_F, env_min_F, env_max_F, color='royalblue', alpha=0.3)
        
    # 绘制额外case的Mar数据（从x=59开始）
    if var_M is not None:
        x_M = range(59, 59 + len(var_M.time))
        mean_M = var_M.mean(dim='member')
        env_min_M = var_M.min(dim='member')
        env_max_M = var_M.max(dim='member')
        ax.plot(x_M, mean_M, color='hotpink', linewidth=3, label='Mar small pertlim')
        ax.fill_between(x_M, env_min_M, env_max_M, color='hotpink', alpha=0.3)
    
    # 添加氯活化阈值线（197K）
    ax.axhline(y=197, color='royalblue', linestyle='--')
    ax.text(5, 194, 'Cl activation threshold', horizontalalignment='left', fontsize=20, color='royalblue')
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=22)
    
    hpa = plev / 100
    ax.set_ylabel(f'Minimum Temperature, {lat_range[0]}-{lat_range[1]}°N, {hpa:.0f} hPa (K)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    plt.legend(fontsize=16)
    
    plt.savefig(f'{save_path}.pdf')
    plt.savefig(f'{save_path}.png')
    plt.close(fig)
    print(f"Saved temperature evolution plot with reference for case {case_id}")
    return fig, ax

def plot_wind_evolution_newcase_with_ref(var_base, var_base_clim, var_F, var_M, case_id, save_path, plev=1000, lat=60):
    """
    绘制额外case的纬向风演变图（仅Feb和Mar），并叠加case特定的参考曲线。
    参数：
      var_base, var_base_clim: 来自merged.nc文件的风数据基准和按日气候态
      var_F, var_M: 对应额外case中Feb和Mar的风数据（由 process_waccm_data 获得）
      case_id: 字符串形式的case编号（例如 '0013'）
      save_path: 保存图形时的文件名前缀
      plev: 氣压层（Pa），默认1000（即10 hPa）
      lat: 选取的纬度，默认60（即60°N）
    """
    sns.set_style("ticks")
    fig, ax = plt.subplots(1, 1, figsize=(12,8), constrained_layout=True, edgecolor='k')
    
    try:
        case_year = int(case_id)
    except Exception as e:
        case_year = None

    ref_curve = None
    ref_clim = None
    if case_year is not None and var_base is not None:
        wind_select = var_base  # 已经在 process_base_data 中选了 1-6 月
        ref_curve = wind_select.sel(time=wind_select.time.dt.year.isin([case_year]))
        ref_clim = var_base_clim
    
    if ref_curve is not None and len(ref_curve.time) > 0:
        x_ref = range(len(ref_curve.time))
        ax.plot(x_ref, ref_curve, color='black', linewidth=3, label=f'Base ({case_year})')
    if ref_clim is not None and len(ref_clim.dayofyear) > 0:
        x_ref_clim = range(len(ref_clim.dayofyear))
        ax.plot(x_ref_clim, ref_clim, color='black', linestyle=':', linewidth=3, label='Clim (all)')
    
    if var_F is not None:
        x_F = range(31, 31 + len(var_F.time))
        mean_F = var_F.mean(dim='member')
        env_min_F = var_F.min(dim='member')
        env_max_F = var_F.max(dim='member')
        ax.plot(x_F, mean_F, color='royalblue', linewidth=3, label='Feb small pertlim')
        ax.fill_between(x_F, env_min_F, env_max_F, color='royalblue', alpha=0.3)
    if var_M is not None:
        x_M = range(59, 59 + len(var_M.time))
        mean_M = var_M.mean(dim='member')
        env_min_M = var_M.min(dim='member')
        env_max_M = var_M.max(dim='member')
        ax.plot(x_M, mean_M, color='hotpink', linewidth=3, label='Mar small pertlim')
        ax.fill_between(x_M, env_min_M, env_max_M, color='hotpink', alpha=0.3)
    
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=22)
    
    hpa = plev / 100
    ax.set_ylabel(f'Zonal mean zonal wind, {lat}°N, {hpa:.0f} hPa (m/s)', fontsize=22)
    ax.tick_params(axis='y', labelsize=22)
    plt.legend(fontsize=16)
    
    plt.savefig(f'{save_path}.pdf')
    plt.savefig(f'{save_path}.png')
    plt.close(fig)
    print(f"Saved wind evolution plot with reference for case {case_id}")
    return fig, ax

# -----------------------------
# 以下循环演示如何处理多个 case
# -----------------------------
new_cases = ['0013', '0014', '0019', '0003']

# merged 文件固定路径
merged_file_path = os.path.join("/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart", "BWCN.e122.f19_g16.merged.nc")

for case in new_cases:
    # case-specific 子目录，用于 Feb 和 Mar 数据
    case_path = f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{case}"
    
    # --------- 温度数据处理 ----------
    try:
        # 返回全部年份(1-6月)的温度数据 (var_temp_base) 及其日气候态 (var_temp_clim)
        var_temp_base, var_temp_clim = process_temp_base_data(merged_file_path, plev=5000, lat_range=(60,90))
    except Exception as e:
        print(f"Error processing base temperature for case {case}: {e}")
        var_temp_base, var_temp_clim = None, None
        
    feb_temp_pattern = os.path.join(case_path, "Feb", f"BWCN.e122.f19_g16.002.{case}-02.*.nc")
    try:
        var_temp_F = process_temp_data(feb_temp_pattern, plev=5000, lat_range=(60,90))
    except Exception as e:
        print(f"Error processing Feb temperature for case {case}: {e}")
        var_temp_F = None
        
    mar_temp_pattern = os.path.join(case_path, "Mar", f"BWCN.e122.f19_g16.002.{case}-03.*.nc")
    try:
        var_temp_M = process_temp_data(mar_temp_pattern, plev=5000, lat_range=(60,90))
    except Exception as e:
        print(f"Error processing Mar temperature for case {case}: {e}")
        var_temp_M = None
        
    temp_save_path = f"/home/weiji/restart_exam/plots/T_evolution_min_restart_{case}_withRef"
    plot_temp_evolution_newcase_with_ref(var_temp_base, var_temp_clim, var_temp_F, var_temp_M,
                                         case, temp_save_path, plev=5000, lat_range=(60,90))
    
    # --------- 风数据处理 ----------
    try:
        var_wind_base, var_wind_clim = process_base_data(merged_file_path, lat=60, plev=1000, months=[1,2,3,4,5,6])
    except Exception as e:
        print(f"Error processing base wind for case {case}: {e}")
        var_wind_base, var_wind_clim = None, None
        
    feb_wind_pattern = os.path.join(case_path, "Feb", f"BWCN.e122.f19_g16.002.{case}-02.*.nc")
    try:
        var_wind_F = process_waccm_data(feb_wind_pattern, lat=60, plev=1000)
    except Exception as e:
        print(f"Error processing Feb wind for case {case}: {e}")
        var_wind_F = None
        
    mar_wind_pattern = os.path.join(case_path, "Mar", f"BWCN.e122.f19_g16.002.{case}-03.*.nc")
    try:
        var_wind_M = process_waccm_data(mar_wind_pattern, lat=60, plev=1000)
    except Exception as e:
        print(f"Error processing Mar wind for case {case}: {e}")
        var_wind_M = None
        
    wind_save_path = f"/home/weiji/restart_exam/plots/U_evolution_min_restart_{case}_withRef"
    plot_wind_evolution_newcase_with_ref(var_wind_base, var_wind_clim, var_wind_F, var_wind_M,
                                         case, wind_save_path, plev=1000, lat=60)


In [ ]:
##############################
# Updated New Code for Normal vs NoCoupl Comparison with Base & Clim Curves
##############################

import os
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

def plot_temp_normal_vs_nocouple(case, 
                                 normal_feb, normal_mar, 
                                 nocouple_feb, nocouple_mar, 
                                 var_base, var_base_clim, 
                                 save_path, plev=5000, lat_range=(60,90)):
    """
    绘制每个 case 中 Feb 和 Mar 普通温度数据与 NoCoupl 温度数据的对比，
    同时叠加该 case 对应的 base (case_year) 曲线和日气候态曲线，
    每个月在一个子图中显示。
    """
    sns.set_style("ticks")
    fig, axs = plt.subplots(1, 2, figsize=(16,8), constrained_layout=True)
    
    # 尝试获取 case_year（用于从 base 数据中提取对应年份的曲线）
    try:
        case_year = int(case)
    except Exception as e:
        case_year = None

    # --------- February subplot ---------
    ax = axs[0]
    # 先绘制基准曲线和气候态曲线（整个时间范围，x坐标从0开始）
    if case_year is not None and var_base is not None:
        ref_curve = var_base.sel(time=var_base.time.dt.year.isin([case_year]))
        if len(ref_curve.time) > 0:
            x_ref = range(len(ref_curve.time))
            ax.plot(x_ref, ref_curve, color='black', linewidth=3, label=f'Base ({case_year})')
    if var_base_clim is not None and hasattr(var_base_clim, "dayofyear") and len(var_base_clim.dayofyear) > 0:
        x_ref_clim = range(len(var_base_clim.dayofyear))
        ax.plot(x_ref_clim, var_base_clim, color='black', linestyle=':', linewidth=3, label='Clim (all)')
        
    # February 普通数据 (偏移 x=31)
    x_normal = range(31, 31 + len(normal_feb.time))
    normal_mean = normal_feb.mean(dim='member')
    normal_min = normal_feb.min(dim='member')
    normal_max = normal_feb.max(dim='member')
    ax.plot(x_normal, normal_mean, color='royalblue', linewidth=3, label='Feb Normal')
    ax.fill_between(x_normal, normal_min, normal_max, color='royalblue', alpha=0.3)
    
    # February NoCoupl 数据 (偏移 x=31)
    x_nocouple = range(31, 31 + len(nocouple_feb.time))
    nocouple_mean = nocouple_feb.mean(dim='member')
    nocouple_min = nocouple_feb.min(dim='member')
    nocouple_max = nocouple_feb.max(dim='member')
    ax.plot(x_nocouple, nocouple_mean, color='hotpink', linestyle='--', linewidth=3, label='Feb NoCoupl')
    ax.fill_between(x_nocouple, nocouple_min, nocouple_max, color='hotpink', alpha=0.3)
    
    # 添加氯活化阈值线（例如197K）
    ax.axhline(y=197, color='gray', linestyle='--', linewidth=2)
    ax.text(5, 194, 'Cl activation threshold', horizontalalignment='left', fontsize=14, color='gray')
    
    ax.set_title(f'Case {case} - February', fontsize=18)
    ax.set_xlabel('Day of Year', fontsize=16)
    ax.set_ylabel(f'Temperature at {plev/100:.0f} hPa, {lat_range[0]}-{lat_range[1]}°N (K)', fontsize=16)
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=14)
    ax.legend(fontsize=12)
    
    # --------- March subplot ---------
    ax = axs[1]
    # 同样绘制基准和气候态曲线
    if case_year is not None and var_base is not None:
        ref_curve = var_base.sel(time=var_base.time.dt.year.isin([case_year]))
        if len(ref_curve.time) > 0:
            x_ref = range(len(ref_curve.time))
            ax.plot(x_ref, ref_curve, color='black', linewidth=3, label=f'Base ({case_year})')
    if var_base_clim is not None and hasattr(var_base_clim, "dayofyear") and len(var_base_clim.dayofyear) > 0:
        x_ref_clim = range(len(var_base_clim.dayofyear))
        ax.plot(x_ref_clim, var_base_clim, color='black', linestyle=':', linewidth=3, label='Clim (all)')
    
    # March 普通数据 (偏移 x=59)
    x_normal = range(59, 59 + len(normal_mar.time))
    normal_mean = normal_mar.mean(dim='member')
    normal_min = normal_mar.min(dim='member')
    normal_max = normal_mar.max(dim='member')
    ax.plot(x_normal, normal_mean, color='navy', linewidth=3, label='Mar Normal')
    ax.fill_between(x_normal, normal_min, normal_max, color='navy', alpha=0.3)
    
    # March NoCoupl 数据 (偏移 x=59)
    x_nocouple = range(59, 59 + len(nocouple_mar.time))
    nocouple_mean = nocouple_mar.mean(dim='member')
    nocouple_min = nocouple_mar.min(dim='member')
    nocouple_max = nocouple_mar.max(dim='member')
    ax.plot(x_nocouple, nocouple_mean, color='deeppink', linestyle='--', linewidth=3, label='Mar NoCoupl')
    ax.fill_between(x_nocouple, nocouple_min, nocouple_max, color='deeppink', alpha=0.3)
    
    ax.axhline(y=197, color='gray', linestyle='--', linewidth=2)
    ax.text(5, 194, 'Cl activation threshold', horizontalalignment='left', fontsize=14, color='gray')
    
    ax.set_title(f'Case {case} - March', fontsize=18)
    ax.set_xlabel('Day of Year', fontsize=16)
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=14)
    ax.legend(fontsize=12)
    
    plt.savefig(f'{save_path}.pdf')
    plt.savefig(f'{save_path}.png')
    plt.close(fig)
    print(f"Saved temperature normal vs NoCoupl comparison plot for case {case}")
    return fig, axs

def plot_wind_normal_vs_nocouple(case, 
                                 normal_feb, normal_mar, 
                                 nocouple_feb, nocouple_mar, 
                                 var_base, var_base_clim, 
                                 save_path, plev=1000, lat=60):
    """
    绘制每个 case 中 Feb 和 Mar 普通风数据与 NoCoupl 风数据的对比，
    同时叠加该 case 对应的 base (case_year) 曲线和日气候态曲线，
    每个月在一个子图中显示。
    """
    sns.set_style("ticks")
    fig, axs = plt.subplots(1, 2, figsize=(16,8), constrained_layout=True)
    
    try:
        case_year = int(case)
    except Exception as e:
        case_year = None

    # --------- February subplot ---------
    ax = axs[0]
    # 绘制基准曲线和气候态曲线
    if case_year is not None and var_base is not None:
        ref_curve = var_base.sel(time=var_base.time.dt.year.isin([case_year]))
        if len(ref_curve.time) > 0:
            x_ref = range(len(ref_curve.time))
            ax.plot(x_ref, ref_curve, color='black', linewidth=3, label=f'Base ({case_year})')
    if var_base_clim is not None and hasattr(var_base_clim, "dayofyear") and len(var_base_clim.dayofyear) > 0:
        x_ref_clim = range(len(var_base_clim.dayofyear))
        ax.plot(x_ref_clim, var_base_clim, color='black', linestyle=':', linewidth=3, label='Clim (all)')
    
    # February 普通数据 (偏移 x=31)
    x_normal = range(31, 31 + len(normal_feb.time))
    normal_mean = normal_feb.mean(dim='member')
    normal_min = normal_feb.min(dim='member')
    normal_max = normal_feb.max(dim='member')
    ax.plot(x_normal, normal_mean, color='royalblue', linewidth=3, label='Feb Normal')
    ax.fill_between(x_normal, normal_min, normal_max, color='royalblue', alpha=0.3)
    
    # February NoCoupl 数据 (偏移 x=31)
    x_nocouple = range(31, 31 + len(nocouple_feb.time))
    nocouple_mean = nocouple_feb.mean(dim='member')
    nocouple_min = nocouple_feb.min(dim='member')
    nocouple_max = nocouple_feb.max(dim='member')
    ax.plot(x_nocouple, nocouple_mean, color='hotpink', linestyle='--', linewidth=3, label='Feb NoCoupl')
    ax.fill_between(x_nocouple, nocouple_min, nocouple_max, color='hotpink', alpha=0.3)
    
    ax.set_title(f'Case {case} - February', fontsize=18)
    ax.set_xlabel('Day of Year', fontsize=16)
    ax.set_ylabel(f'Zonal mean wind at {lat}°N, {plev/100:.0f} hPa (m/s)', fontsize=16)
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=14)
    ax.legend(fontsize=12)
    
    # --------- March subplot ---------
    ax = axs[1]
    # 绘制基准曲线和气候态曲线
    if case_year is not None and var_base is not None:
        ref_curve = var_base.sel(time=var_base.time.dt.year.isin([case_year]))
        if len(ref_curve.time) > 0:
            x_ref = range(len(ref_curve.time))
            ax.plot(x_ref, ref_curve, color='black', linewidth=3, label=f'Base ({case_year})')
    if var_base_clim is not None and hasattr(var_base_clim, "dayofyear") and len(var_base_clim.dayofyear) > 0:
        x_ref_clim = range(len(var_base_clim.dayofyear))
        ax.plot(x_ref_clim, var_base_clim, color='black', linestyle=':', linewidth=3, label='Clim (all)')
    
    # March 普通数据 (偏移 x=59)
    x_normal = range(59, 59 + len(normal_mar.time))
    normal_mean = normal_mar.mean(dim='member')
    normal_min = normal_mar.min(dim='member')
    normal_max = normal_mar.max(dim='member')
    ax.plot(x_normal, normal_mean, color='navy', linewidth=3, label='Mar Normal')
    ax.fill_between(x_normal, normal_min, normal_max, color='navy', alpha=0.3)
    
    # March NoCoupl 数据 (偏移 x=59)
    x_nocouple = range(59, 59 + len(nocouple_mar.time))
    nocouple_mean = nocouple_mar.mean(dim='member')
    nocouple_min = nocouple_mar.min(dim='member')
    nocouple_max = nocouple_mar.max(dim='member')
    ax.plot(x_nocouple, nocouple_mean, color='deeppink', linestyle='--', linewidth=3, label='Mar NoCoupl')
    ax.fill_between(x_nocouple, nocouple_min, nocouple_max, color='deeppink', alpha=0.3)
    
    ax.set_title(f'Case {case} - March', fontsize=18)
    ax.set_xlabel('Day of Year', fontsize=16)
    ax.set_xlim(0,150)
    ax.set_xticks([0,31,59,89,120])
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','June'], fontsize=14)
    ax.legend(fontsize=12)
    
    plt.savefig(f'{save_path}.pdf')
    plt.savefig(f'{save_path}.png')
    plt.close(fig)
    print(f"Saved wind normal vs NoCoupl comparison plot for case {case}")
    return fig, axs

# -----------------------------
# 循环处理 nocouple 数据（仅对 0013, 0014, 0008, 0019 这几个 case）
# -----------------------------
nocouple_cases = ['0013', '0014', '0008', '0019']

# merged 文件固定路径（与之前一致）
merged_file_path = os.path.join("/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart", "BWCN.e122.f19_g16.merged.nc")

for case in nocouple_cases:
    # 普通数据存放目录（与原有代码一致）
    normal_case_path = f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{case}"
    
    normal_feb_pattern = os.path.join(normal_case_path, "Feb", f"BWCN.e122.f19_g16.002.{case}-02.*.nc")
    normal_mar_pattern = os.path.join(normal_case_path, "Mar", f"BWCN.e122.f19_g16.002.{case}-03.*.nc")
    
    # NoCoupl 数据存放目录（按照给定的目录结构）
    nocouple_feb_pattern = os.path.join("/home/weiji/restart_exam/nocouple_data", case, "Feb_NOCOUPL", f"{case}-02", "*.nc")
    nocouple_mar_pattern = os.path.join("/home/weiji/restart_exam/nocouple_data", case, "Mar_NOCOUPL", f"{case}-03", "*.nc")
    
    # 处理温度数据
    try:
        normal_temp_F = process_temp_data(normal_feb_pattern, plev=5000, lat_range=(60,90))
    except Exception as e:
        print(f"Error processing normal Feb temperature for case {case}: {e}")
        normal_temp_F = None
    try:
        normal_temp_M = process_temp_data(normal_mar_pattern, plev=5000, lat_range=(60,90))
    except Exception as e:
        print(f"Error processing normal Mar temperature for case {case}: {e}")
        normal_temp_M = None
    try:
        nocouple_temp_F = process_temp_data(nocouple_feb_pattern, plev=5000, lat_range=(60,90))
    except Exception as e:
        print(f"Error processing NoCoupl Feb temperature for case {case}: {e}")
        nocouple_temp_F = None
    try:
        nocouple_temp_M = process_temp_data(nocouple_mar_pattern, plev=5000, lat_range=(60,90))
    except Exception as e:
        print(f"Error processing NoCoupl Mar temperature for case {case}: {e}")
        nocouple_temp_M = None
        
    # 读取 merged 文件的温度基准数据及其日气候态（对所有年份，后续在绘图中筛选对应 case_year）
    try:
        var_temp_base, var_temp_clim = process_temp_base_data(merged_file_path, plev=5000, lat_range=(60,90))
    except Exception as e:
        print(f"Error processing base temperature for case {case}: {e}")
        var_temp_base, var_temp_clim = None, None
        
    temp_compare_save_path = f"/home/weiji/restart_exam/plots/T_evolution_normal_vs_nocouple_{case}"
    if (normal_temp_F is not None and normal_temp_M is not None and
        nocouple_temp_F is not None and nocouple_temp_M is not None and
        var_temp_base is not None and var_temp_clim is not None):
        plot_temp_normal_vs_nocouple(case, normal_temp_F, normal_temp_M, 
                                     nocouple_temp_F, nocouple_temp_M, 
                                     var_temp_base, var_temp_clim,
                                     temp_compare_save_path, plev=5000, lat_range=(60,90))
    
    # 处理风数据
    try:
        normal_wind_F = process_waccm_data(normal_feb_pattern, lat=60, plev=1000)
    except Exception as e:
        print(f"Error processing normal Feb wind for case {case}: {e}")
        normal_wind_F = None
    try:
        normal_wind_M = process_waccm_data(normal_mar_pattern, lat=60, plev=1000)
    except Exception as e:
        print(f"Error processing normal Mar wind for case {case}: {e}")
        normal_wind_M = None
    try:
        nocouple_wind_F = process_waccm_data(nocouple_feb_pattern, lat=60, plev=1000)
    except Exception as e:
        print(f"Error processing NoCoupl Feb wind for case {case}: {e}")
        nocouple_wind_F = None
    try:
        nocouple_wind_M = process_waccm_data(nocouple_mar_pattern, lat=60, plev=1000)
    except Exception as e:
        print(f"Error processing NoCoupl Mar wind for case {case}: {e}")
        nocouple_wind_M = None
        
    # 读取 merged 文件的风数据基准及其气候态
    try:
        var_wind_base, var_wind_clim = process_base_data(merged_file_path, lat=60, plev=1000, months=[1,2,3,4,5,6])
    except Exception as e:
        print(f"Error processing base wind for case {case}: {e}")
        var_wind_base, var_wind_clim = None, None
        
    wind_compare_save_path = f"/home/weiji/restart_exam/plots/U_evolution_normal_vs_nocouple_{case}"
    if (normal_wind_F is not None and normal_wind_M is not None and
        nocouple_wind_F is not None and nocouple_wind_M is not None and
        var_wind_base is not None and var_wind_clim is not None):
        plot_wind_normal_vs_nocouple(case, normal_wind_F, normal_wind_M, 
                                     nocouple_wind_F, nocouple_wind_M, 
                                     var_wind_base, var_wind_clim,
                                     wind_compare_save_path, plev=1000, lat=60)
